# Práctica 1 — Dataset 3: Barrios analíticos ↔ Census Tracts (Bronze ➜ Silver)

**Objetivo:** Leer el CSV de relación entre barrios analíticos y *ensus tracts desde bronze,
quedarnos con las columnas necesarias, validar la clave geográfica (`census_tract`) y persistir en silver como Parquet.

In [1]:
# ============================================================
# 0) Imports + SparkSession
# ============================================================
from pyspark.sql import SparkSession, functions as F, types as T
import os

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

print("Spark version:", spark.version)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /opt/spark/ivy/cache
The jars for the packages stored in: /opt/spark/ivy/jars
org.apache.spark#spark-hadoop-cloud_2.13 added as a dependency
io.delta#delta-spark_2.13 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5d8ef414-5cc2-42d6-8be4-7a02294a2486;1.0
	confs: [default]
	found org.apache.spark#spark-hadoop-cloud_2.13;4.0.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
	found s

Spark version: 4.0.1


In [2]:
# Funcion para facilitar visualización
from IPython.display import display
import pandas as pd
pd.set_option("display.max_columns", None)  # muestra todas las columnas

def display_df(
    df,
    n=10,
    columns=None,
    sort_by=None,
    ascending=False
):
    """
    Visualiza un DataFrame de Spark en Jupyter de forma similar a Databricks display().

    Parámetros:
    - df: DataFrame de Spark
    - n: número máximo de filas a mostrar (default: 20)
    - columns: lista opcional de columnas a mostrar
    - sort_by: columna opcional para ordenar
    - ascending: orden ascendente o descendente
    """

    if columns:
        df = df.select(columns)

    if sort_by:
        df = df.orderBy(sort_by, ascending=ascending)

    pdf = df.limit(n).toPandas()
    display(pdf)


In [3]:
# ============================================================
# 0.1) Configuración MinIO (S3A)
# ============================================================
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minioadmin")
MINIO_USE_SSL = os.getenv("MINIO_USE_SSL", "false").lower() == "true"

hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", MINIO_ENDPOINT)
hconf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hconf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "true" if MINIO_USE_SSL else "false")
hconf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

print("MINIO_ENDPOINT =", MINIO_ENDPOINT)

MINIO_ENDPOINT = http://minio:9000


## 1) Lectura desde Bronze (sin inferir esquema)

Leemos desde la carpeta del dataset.

In [4]:
BRONZE_PATH = "s3a://bronze/2020_census_tracts_to_neighborhoods_20251208.csv/"

df_raw = (
    spark.read
        .option("header", True)
        .option("inferSchema", "false")
        .csv(BRONZE_PATH)
)

df_raw.printSchema()
display_df(df_raw, n=5)


root
 |-- the_geom: string (nullable = true)
 |-- object_id: string (nullable = true)
 |-- state_fp: string (nullable = true)
 |-- county_fp: string (nullable = true)
 |-- tractce: string (nullable = true)
 |-- name: string (nullable = true)
 |-- neighborhoods_analysis_boundaries: string (nullable = true)
 |-- data_loaded_at: string (nullable = true)
 |-- sup_dist_2012: string (nullable = true)
 |-- sup_dist_2022: string (nullable = true)
 |-- data_as_of: string (nullable = true)
 |-- geoid: string (nullable = true)



,the_geom,object_id,state_fp,county_fp,tractce,name,neighborhoods_analysis_boundaries,data_loaded_at,sup_dist_2012,sup_dist_2022,data_as_of,geoid
0,MULTIPOLYGON (((-122.37276211607421 37.7455051...,242,06,075,980900,9809,Bayview Hunters Point,2022 Jul 08 02:12:00 PM,10,10,2022 Jul 08 09:09:48 PM,06075980900
1,MULTIPOLYGON (((-122.36519199065519 37.7337298...,241,06,075,980600,9806,Bayview Hunters Point,2022 Jul 08 02:12:00 PM,10,10,2022 Jul 08 09:09:48 PM,06075980600
2,MULTIPOLYGON (((-122.40666500014143 37.7192149...,240,06,075,980501,9805.01,McLaren Park,2022 Jul 08 02:12:00 PM,10,10,2022 Jul 08 09:09:48 PM,06075980501
3,MULTIPOLYGON (((-123.00359900126116 37.6932479...,239,06,075,980401,9804.01,The Farallones,2022 Jul 08 02:12:00 PM,1,4,2022 Jul 08 09:09:48 PM,06075980401
4,MULTIPOLYGON (((-122.38528200051307 37.7402399...,226,06,075,061200,612,Bayview Hunters Point,2022 Jul 08 02:12:00 PM,10,10,2022 Jul 08 09:09:48 PM,06075061200


## 2) Seleccionar y renombrar columnas en un solo paso

- `neighborhoods_analysis_boundaries` → `analysis_neighborhood`
- `geoid` → `census_tract`

In [5]:
df = df_raw.select(
    F.col("neighborhoods_analysis_boundaries").alias("analysis_neighborhood"),
    F.col("geoid").alias("census_tract"),
)

df.printSchema()
display_df(df, n=10)


root
 |-- analysis_neighborhood: string (nullable = true)
 |-- census_tract: string (nullable = true)



,analysis_neighborhood,census_tract
0,Bayview Hunters Point,06075980900
1,Bayview Hunters Point,06075980600
2,McLaren Park,06075980501
3,The Farallones,06075980401
4,Bayview Hunters Point,06075061200
5,Chinatown,06075061102
6,Chinatown,06075061101
7,Bayview Hunters Point,06075061000
8,Golden Gate Park,06075980300
9,Outer Richmond,06075047600


## 3) Validación de la clave geográfica `census_tract`

Comprobaciones requeridas:

1. Es de tipo **string**
2. Tiene longitud **11**
3. Conserva correctamente los **ceros a la izquierda**

En las siguientes celdas se realizan las comprobaciones y se muestran métricas para justificar el resultado.

In [6]:
from pyspark.sql.types import StringType

# 1) Tipo string
is_string = isinstance(df.schema["census_tract"].dataType, StringType)
print("¿census_tract es string?:", is_string)

# 2) Longitud 11
invalid_len = df.filter(F.length("census_tract") != 11).count()
print("Filas con longitud != 11:", invalid_len)

# 3) Ceros a la izquierda:
# Si la columna se hubiese leído como numérica, valores como 06075010101 perderían el 0 inicial.
# Medimos cuántos valores empiezan por '0' y además verificamos que son 11 dígitos.
starts_with_zero = df.filter(F.col("census_tract").startswith("0")).count()
not_11_digits = df.filter(~F.col("census_tract").rlike(r"^\d{11}$")).count()

print("Filas cuyo census_tract empieza por '0':", starts_with_zero)
print("Filas cuyo census_tract NO cumple ^\d{11}$:", not_11_digits)

# Muestra ejemplos (incluye potencialmente valores con 0 inicial)
display_df(df.select("census_tract").orderBy("census_tract"), n=10)


¿census_tract es string?: True
Filas con longitud != 11: 0
Filas cuyo census_tract empieza por '0': 242
Filas cuyo census_tract NO cumple ^\d{11}$: 0


,census_tract
0,06075010101
1,06075010102
2,06075010201
3,06075010202
4,06075010300
5,06075010401
6,06075010402
7,06075010500
8,06075010600
9,06075010701


**Explicación (completa tras ejecutar):**

- **Tipo:** `True` → se mantiene como `string` (al leer CSV sin inferir esquema).
- **Longitud:** `0` filas con longitud distinta de 11 → **correcto** (todas tienen 11 dígitos).
- **Ceros a la izquierda:** hay `242` valores que empiezan por `0` y `0` filas que no cumplen `^\d{11}$` → los ceros se han preservado.

Conclusión: `census_tract` es una clave geográfica válida para unir con otros datasets a nivel de Census Tract.


## 4) Escritura en Silver (Parquet)

Persistimos el resultado en:

- `silver/sf_neighborhoods_census_tracts/`

In [7]:
SILVER_PATH = "s3a://silver/sf_neighborhoods_census_tracts/"

(
    df.write
      .mode("overwrite")
      .parquet(SILVER_PATH)
)

df_check = spark.read.parquet(SILVER_PATH)
print("Filas en silver:", df_check.count())
df_check.printSchema()

26/03/03 11:43:00 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to sf_neighborhoods_census_tracts/_temporary/0/_temporary/attempt_202603031142595548039495050977236_0013_m_000000_10/part-00000-ed3c42dd-1864-46e7-9e90-4652524e6887-c000.snappy.parquet. This is Unsupported
                                                                                

Filas en silver: 242
root
 |-- analysis_neighborhood: string (nullable = true)
 |-- census_tract: string (nullable = true)



## 5) Preguntas analíticas

### i) ¿Cuántos census tracts distintos hay?

In [8]:
n_tracts = df.select("census_tract").distinct().count()
print(n_tracts)

242


Hay 242 census tracts distintos.


### ii) ¿Cuántos barrios analíticos distintos hay?

In [9]:
n_neigh = df.select("analysis_neighborhood").distinct().count()
print(n_neigh)

42


Hay 42 barrios analíticos distintos.


### iii) ¿Puede un barrio estar asociado a varios census tracts?

In [10]:
neigh_stats = (
    df.groupBy("analysis_neighborhood")
      .agg(F.countDistinct("census_tract").alias("n_tracts"))
)

neigh_with_many = neigh_stats.filter(F.col("n_tracts") > 1).count()
max_tracts_per_neigh = neigh_stats.agg(F.max("n_tracts").alias("max_n_tracts")).collect()[0]["max_n_tracts"]

print(neigh_with_many)
print(max_tracts_per_neigh)

34
17


Si, hay 34 barrios con más de 1 census tract (el número máximo de tracts asociados a un mismo barrio es de 17).


### iv) ¿Puede un census tract pertenecer a varios barrios?

In [11]:
tract_stats = (
    df.groupBy("census_tract")
      .agg(F.countDistinct("analysis_neighborhood").alias("n_neigh"))
)

tracts_with_many = tract_stats.filter(F.col("n_neigh") > 1).count()
max_neigh_per_tract = tract_stats.agg(F.max("n_neigh").alias("max_n_neigh")).collect()[0]["max_n_neigh"]

print(tracts_with_many)
print(max_neigh_per_tract)

0
1


No. Hay 0 census tracts asociados a más de un barrio (y obviamente el máximo de barrios por tract es 1).


### v) Justifica brevemente qué tipo de relación existe entre barrios analíticos y census tracts

Usando los resultados de (iii) y (iv) para justificar si la relación es:
- 1 a 1
- 1 a N
- N a 1
- N a M

In [12]:
# Calculamos indicadores booleanos
can_neigh_have_many_tracts = True #Los vecindarios pueden tener mas de un census tract como hemos visto en el ejercicio iii.)
can_tract_have_many_neigh = False #Los census tract solo pueden pertenecer a un barrio, como hemos visto en el ejercicio iv.)

#Viendo todos los casos posibles, vamos a ver cual sale como verídico.
if can_neigh_have_many_tracts and can_tract_have_many_neigh:
    relation = "N a M"
elif can_neigh_have_many_tracts and not can_tract_have_many_neigh:
    relation = "1 a N"
elif (not can_neigh_have_many_tracts) and can_tract_have_many_neigh:
    relation = "N a 1"
else:
    relation = "1 a 1"

print(relation)

1 a N


Tipo de relación: 1 a N (de *barrio* → *census tracts*). Segun las dos celdas anteriores, al haber 34 barrios que agrupan varios tracts, pero ningún tract pertenece a más de un barrio podemos decir que la relacion es de un barrio a muchos tracts. La lógica del codigo de arriba es exactamente la misma que la descrita en esta celda.
